In [89]:
import numpy as np
import pandas as pd

In [90]:
def extract_refined_features(matrix):
    # time = matrix[:, 0]
    # Assuming columns are: [time, p4, p5, p6]
    p5 = matrix[:, 2]
    p6 = matrix[:, 3]
    
    # 1. Baseline Correction (using first 20 samples)
    p5_zeroed = p5 - np.mean(p5[:20])
    p6_zeroed = p6 - np.mean(p6[:20])
    
    # 2. Peak Magnitudes
    p5_max = np.max(p5_zeroed)
    p6_max = np.max(p6_zeroed)
    
    # 3. Spatial Spread Ratio (p5 contribution relative to p6)
    # Less rigid materials spread more.
    spread_ratio = p5_max / p6_max if p6_max > 10 else 0
    
    # 4. Area Ratio (Width/Fatness of the peak)
    # We use the primary sensor (p6) for this.
    area = np.trapz(np.maximum(p6_zeroed, 0)) # Integral of positive part
    area_ratio = area / p6_max if p6_max > 10 else 0
    
    return {
        "max_p": p6_max,
        "ratio": spread_ratio,
        "area": area_ratio
    }

def classify_sample(matrix):
    feat = extract_refined_features(matrix)
    p = feat['max_p']
    r = feat['ratio']
    a = feat['area']
    
    # --- Classification Logic ---
    
    # 1. Identify Sample A (Low Peak Magnitude)
    if p < 200:
        return "Sample A (Least Rigid)"
    
    # 2. Identify Sample F (Highest Peak + High Spatial Spread)
    # Rigid F hits multiple sensors hard due to structural transmission.
    if p > 800 or r > 0.2:
        return "Sample F (Most Rigid)"
    
    # 3. Distinguish D vs E (The "Spread" Test)
    # Sample D (less rigid) deforms and spreads to p5 more than E.
    if r > 0.04:
        return "Sample D"
    else:
        return "Sample E"

In [116]:
df = pd.read_csv('SampleF_Case3.csv')
matrix = df[['time', 'p4', 'p5', 'p6']].values
print(classify_sample(matrix))

Sample F (Most Rigid)
